In [2]:
import pandas as pd

final = pd.read_parquet("../data/processed/modeling_table_route607.parquet")
final_clean = final.dropna(subset=["is_delayed"]).copy()

print(f"{len(final_clean)} rows across {final_clean['service_date'].nunique()} dates")

240665 rows across 54 dates


In [4]:
majority_baseline = final_clean["is_delayed"].value_counts(normalize=True).max()
print(f"Majority-class baseline accuracy: {majority_baseline:.4f}")
print(f"Overall delay rate: {final_clean['is_delayed'].mean():.4f}")

Majority-class baseline accuracy: 0.7705
Overall delay rate: 0.2295


Baselines (route 607, 54 dates, 240,665 rows): majority-class baseline is 77.05% accuracy (all-not-delayed). Route 607's overall winter delay rate is 22.95%, notably lower than the system-wide winter figure of 27.9% found in the broader 23-date EDA, suggesting this specific commute route performs somewhat better than SL's winter average.

In [5]:
info = final_clean.groupby("service_date").agg(
    mean_temp=("temperature_c", "mean"),
    n_rows=("is_delayed", "size"),
    delay_rate=("is_delayed", "mean"),
)
info["weekday"] = pd.to_datetime(info.index).day_name()
print(info.sort_values("mean_temp"))

              mean_temp  n_rows  delay_rate    weekday
service_date                                          
2024-01-16   -12.532756    5080    0.368504    Tuesday
2023-12-06   -11.974875    5202    0.415033  Wednesday
2024-01-19   -11.839976    5083    0.400944     Friday
2021-12-06   -11.209159    5110    0.229159     Monday
2021-12-07   -10.506239    5129    0.270618    Tuesday
2026-02-14    -9.025725    3071    0.165418   Saturday
2022-12-15    -8.993469    5007    0.316357   Thursday
2023-12-05    -8.883641    5202    0.330834    Tuesday
2021-02-10    -7.647628    5123    0.234238  Wednesday
2021-01-15    -7.600890    4945    0.269565     Friday
2026-02-06    -7.441459    5099    0.108845     Friday
2024-01-18    -6.977606    5055    0.429278   Thursday
2026-02-19    -6.761640    5099    0.151402   Thursday
2021-02-06    -6.665211    3228    0.382900   Saturday
2026-02-12    -6.264234    5083    0.120598   Thursday
2022-12-08    -6.178739    5202    0.239331   Thursday
2021-11-30

weekday dates run ~5000-5250 rows, weekend dates ~2400-3250, confirming 607 does run reduced weekend service (similar to the metro/bus pattern from Phase 3, though 607 still runs some weekend service, unlike being fully absent). And delay rates are genuinely noisy day to day (from 6.7% to 43%), which is exactly why testing on multiple dates matters more than any single one.

In [6]:
test_dates = [
    "2024-01-16",  # -12.5C, coldest available, 2024
    "2021-12-07",  # -10.5C, cold, 2021
    "2026-02-14",  # -9.0C, cold, Saturday, 2026
    "2022-12-15",  # -9.0C, cold-mid, 2022
    "2021-02-10",  # -7.6C, mid-cold, 2021
    "2023-12-12",  # -2.9C, mild-cold, 2023
    "2024-02-20",  #  0.2C, mild, 2024
    "2025-01-31",  #  1.3C, mild, 2025
    "2025-03-23",  #  4.4C, warm, Sunday, 2025
    "2026-02-27",  #  7.1C, warm, 2026
    "2025-03-29",  # 10.2C, warmest available, Saturday, 2025
]

train = final_clean[~final_clean["service_date"].isin(test_dates)]
test = final_clean[final_clean["service_date"].isin(test_dates)]

print(f"Train: {len(train)} rows across {train['service_date'].nunique()} dates")
print(f"Test: {len(test)} rows across {test['service_date'].nunique()} dates")
print(f"Test share: {len(test) / (len(train) + len(test)):.3f}")
print()
print("Train delay rate:", train["is_delayed"].mean())
print("Test delay rate:", test["is_delayed"].mean())
print()
print("Train temp range:", train["temperature_c"].min(), "to", train["temperature_c"].max())
print("Test temp range:", test["temperature_c"].min(), "to", test["temperature_c"].max())

Train: 190903 rows across 43 dates
Test: 49762 rows across 11 dates
Test share: 0.207

Train delay rate: 0.22690581080443994
Test delay rate: 0.2392990635424621

Train temp range: -14.0 to 14.6
Test temp range: -14.2 to 13.3


**Train/test split (route 607)**: 11 dates held out as test (~20.7%), spanning the full temperature range 
(-14.2°C to 13.3°C, closely matching train's -14.0°C to 14.6°C) and including 3 weekend dates. Delay rates 
are close between train (22.7%) and test (23.9%), suggesting neither set is systematically easier/harder.